In [1]:
# Cell 1: 导入依赖 & 配置参数
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import re
import psycopg2
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ---------- 数据库连接配置 ----------
DB_CONFIG = {
    "dbname": "Law_app",
    "user": "my_pgsql",
    "password": "123123",
    "host": "localhost",
    "port": 5433,
}

# ---------- 嵌入模型 ----------
MODEL_NAME = "BAAI/bge-large-zh-v1.5"
model = SentenceTransformer(MODEL_NAME)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
DB_CONFIG = {
    "dbname": "Law_app",
    "user": "my_pgsql",
    "password": "123123",
    "host": "localhost",
    "port": 5433,
}
# 测试数据库连接
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT version();")
    version = cur.fetchone()
    print(f"数据库连接成功！PostgreSQL 版本: {version[0]}")
    cur.close()
    conn.close()
except psycopg2.OperationalError as e:
    print(f"数据库连接失败: {e}")
except Exception as e:
    print(f"未知错误: {e}")

数据库连接成功！PostgreSQL 版本: PostgreSQL 15.4 (Debian 15.4-2.pgdg120+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit


In [4]:
def parse_law_from_file(file_path: str):
    """
    从法律文本文件中解析法条。
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict] 每个元素包含 chapter, article_number, content
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ---------- 匹配章/节标题 ----------
    # 支持三种格式: "第X章 XXX"、"X、XXX" 或无章节
    chapter_pattern = re.compile(
        r"^(?:第([一二三四五六七八九十百千零]+)章\s*(.*))"
        r"|^(?:([一二三四五六七八九十百千零]+)、(.+))",
        re.MULTILINE,
    )
    chapter_matches = list(chapter_pattern.finditer(text))

    if not chapter_matches:
        sections = [("", 0, len(text))]
    else:
        sections = []
        for i, m in enumerate(chapter_matches):
            if m.group(1):  # "第X章 XXX" 格式
                chapter_name = f"第{m.group(1)}章 {m.group(2).strip()}"
            else:  # "X、XXX" 格式
                chapter_name = f"{m.group(3)}、{m.group(4).strip()}"
            start = m.start()
            end = chapter_matches[i + 1].start() if i + 1 < len(chapter_matches) else len(text)
            sections.append((chapter_name, start, end))

    # ---------- 匹配条文起始位置 ----------
    article_start_re = re.compile(r"第([一二三四五六七八九十百千零]+)条\s*")

    # ---------- 过滤施行日期条款 ----------
    _DATE_CLAUSE_RE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def is_effective_date_clause(content: str) -> bool:
        c = content.strip()
        return len(c) < 80 and bool(_DATE_CLAUSE_RE.search(c))

    articles = []
    for section_name, sec_start, sec_end in sections:
        section_text = text[sec_start:sec_end]
        # 找到该章节内所有"第X条"的位置
        article_starts = list(article_start_re.finditer(section_text))

        for i, m in enumerate(article_starts):
            article_num = m.group(1)
            content_start = m.end()  # "第X条"之后
            # 内容区间: 当前条文起始 到 下一条文起始(或章节末尾)
            if i + 1 < len(article_starts):
                content_end = article_starts[i + 1].start()
            else:
                content_end = len(section_text)

            raw_content = section_text[content_start:content_end].strip()

            # 以中文句号作为法条内容的自然结束边界
            last_period = raw_content.rfind("。")
            if last_period != -1:
                raw_content = raw_content[:last_period + 1]

            if not raw_content or is_effective_date_clause(raw_content):
                continue

            articles.append(
                {
                    "chapter": section_name or "",
                    "article_number": f"第{article_num}条",
                    "content": raw_content,
                }
            )

    return law_title, articles

In [10]:
# Cell 3: 数据库建表（首次运行执行一次, 表已存在则跳过）
def create_table_if_not_exists():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS marriage_law (
            id SERIAL PRIMARY KEY,
            law_title TEXT NOT NULL,
            chapter TEXT,
            article_number TEXT NOT NULL,
            content TEXT NOT NULL,
            embedding VECTOR(1024),
            UNIQUE(law_title, article_number)
        );
        -- 索引按需创建, 见 create-schema-template.sql
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("表已就绪。")


create_table_if_not_exists()

表已就绪。


In [12]:
# 创建 agent_memory 表 (长期记忆 — PostgreSQL + pgvector)
def create_memory_table():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS agent_memory (
            id          SERIAL PRIMARY KEY,
            thread_id   TEXT NOT NULL DEFAULT 'default',
            memory_type TEXT NOT NULL DEFAULT 'general',
            content     TEXT NOT NULL,
            embedding   VECTOR(1024),
            metadata    JSONB DEFAULT '{}'::jsonb,
            created_at  TIMESTAMP DEFAULT NOW()
        );
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("agent_memory 表已就绪。")

create_memory_table()

agent_memory 表已就绪。


In [5]:
# Cell 4: 向量化并插入数据库
def insert_articles(law_title: str, articles: list[dict]):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    sql = """
        INSERT INTO marriage_law (law_title, chapter, article_number, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (law_title, article_number) DO UPDATE
        SET content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            chapter = EXCLUDED.chapter
    """
    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
            ),
        )
    conn.commit()
    cur.close()
    conn.close()
    print(f"成功插入/更新 {len(articles)} 条记录。")

In [6]:
def insert_law_vector(law_title: str, articles: list[dict]):
    """
    将解析后的法条数据插入 law_vector 表。
    Args:
        law_title: 法律名称
        articles: parse_law_from_file 返回的法条列表
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    sql = """
        INSERT INTO law_vector (law_title, chapter, article_number, content, embedding)
        SELECT %s, %s, %s, %s, %s
        WHERE NOT EXISTS (
            SELECT 1 FROM law_vector
            WHERE law_title = %s AND article_number = %s
        )
    """
    inserted = 0
    skipped = 0

    for art in articles:
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(
            sql,
            (
                law_title,
                art.get("chapter", ""),
                art["article_number"],
                art["content"],
                embedding,
                law_title,
                art["article_number"],
            ),
        )
        if cur.rowcount > 0:
            inserted += 1
        else:
            skipped += 1

    conn.commit()
    cur.close()
    conn.close()
    print(f"law_vector: 成功插入 {inserted} 条, 跳过 {skipped} 条(已存在)。")

In [7]:
from pathlib import Path

law_dir = Path(r"E:\\LangChain\\lawApp_LangGraph\\Documents\\LawDocument")
txt_files = sorted(law_dir.glob("*.txt"))

if not txt_files:
    print(f"目录 {law_dir} 下未找到 .txt 文件。")
else:
    print(f"共发现 {len(txt_files)} 个法律文件:\n")
    for fp in txt_files:
        print(f"  - {fp.name}")

    # for fp in txt_files:
    #     print(f"\n处理: {fp.name}")
    #     law_title, articles = parse_law_from_file(str(fp))
    #     print(f"  解析到 {len(articles)} 条")
    #     insert_law_vector(law_title, articles)

共发现 3 个法律文件:

  - 中华人民共和国反家庭暴力法.txt
  - 中华人民共和国妇女权益保障法.txt
  - 最高人民法院关于审理涉彩礼纠纷案件.txt


In [8]:
def parse_law_from_file_Version2(file_path: str):
    """
    解析法律文本文件。
    逻辑:
        1. 按章节拆分全文 (第X章 / X、格式 / 无章节则整篇)
        2. 每章内: 以行首"第XX条"为起点, 以中文句号"。"为终点提取条文
        3. 条文内部的"第XX条"引用不参与切分
    Args:
        file_path: 法律文件路径
    Returns:
        law_title: 法律名称
        articles: list[dict]
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    law_title = Path(file_path).stem

    # ── 阶段一: 拆分章节 ──────────────────────────────
    marks = []  # [(pos, name)]

    # 格式A — "第X章 XXX"  (允许前导空格, 如"   第一章 基本 规定")
    for m in re.finditer(
        r"^\s*第([一二三四五六七八九十百千零]+)章\s*(.+)", text, re.MULTILINE
    ):
        marks.append((m.start(), f"第{m[1]}章 {m[2].strip()}"))

    # 格式B — "X、XXX"  (独立短行, ≤40字, 避免匹配文内子项)
    # for m in re.finditer(
    #         r"^\s*([一二三四五六七八九十百千零]+)、(.{1,40})$",
    #         text, re.MULTILINE):
    #     if not any(abs(m.start() - p) < 2 for p, _ in marks):
    #         marks.append((m.start(), f"{m[1]}、{m[2].strip()}"))

    marks.sort(key=lambda x: x[0])

    sections = []
    if not marks:
        sections = [("", 0, len(text))]
    else:
        for i, (start, name) in enumerate(marks):
            end = marks[i + 1][0] if i + 1 < len(marks) else len(text)
            sections.append((name, start, end))

    # ── 阶段二: 逐章提取条文 ────────────────────────────
    # 条文起点: 行首 "第XX条"  (同样允许前导空格)
    HEAD = re.compile(r"^\s*第([一二三四五六七八九十百千零]+)条\s*", re.MULTILINE)

    # 施行日期子句 (不视为法条)
    DATE_CLAUSE = re.compile(
        r"(?:本条例|本规定|本法|本解释)自\d{4}年\d{1,2}月\d{1,2}日起施行"
    )

    def _skip(content: str) -> bool:
        c = content.strip()
        return not c or (len(c) < 80 and DATE_CLAUSE.search(c))

    articles = []
    for idx, (sec_name, sec_start, sec_end) in enumerate(sections, 1):
        zone = text[sec_start:sec_end]
        heads = list(HEAD.finditer(zone))

        for i, h in enumerate(heads):
            num = h[1]
            body_start = h.end()
            body_end = heads[i + 1].start() if i + 1 < len(heads) else len(zone)
            body = zone[body_start:body_end].strip()

            cut = body.rfind("。")
            if cut != -1:
                body = body[: cut + 1]

            if _skip(body):
                continue

            articles.append(
                {
                    "chapter": sec_name or "",
                    "article_number": f"第{num}条",
                    "content": body,
                }
            )

        print(
            f"\r  解析进度: {idx}/{len(sections)} 章节, 已收集 {len(articles)} 条",
            end="",
        )

    print()
    return law_title, articles

In [ ]:
# 批量解析并入库: Error 目录下所有法律文件
from pathlib import Path

law_dir = Path(r"E:\LangChain\lawApp_LangGraph\Documents\LawDocument")
txt_files = sorted(law_dir.glob("*.txt"))

print(f"目录: {law_dir}")
print(f"共发现 {len(txt_files)} 个文件:\n")
for fp in txt_files:
    print(f"  - {fp.name}")

print()

for fp in txt_files:
    print(f"\n{'='*50}")
    print(f"文件: {fp.name}")

    law_title, articles = parse_law_from_file_Version2(str(fp))
    total = len(articles)

    if not articles:
        print("  未解析到任何法条, 跳过")
        continue

    sql = """
        INSERT INTO law_vector (law_title, chapter, article_number, content, embedding)
        VALUES (%s, %s, %s, %s, %s)
    """
    for i, art in enumerate(articles, 1):
        embedding = model.encode(art["content"], normalize_embeddings=True).tolist()
        cur.execute(sql, (
            law_title,
            art.get("chapter", ""),
            art["article_number"],
            art["content"],
            embedding,
        ))
        if i % 50 == 0 or i == total:
            conn.commit()
            print(f"\r  入库: {i}/{total}", end="")

    conn.commit()
    cur.close()
    conn.close()
    print(f"\n  完成: 清除 {deleted} 条旧记录, 插入 {total} 条")


目录: E:\LangChain\lawApp_LangGraph\Documents\LawDocument\Error
共发现 3 个文件:

  - 婚姻登记条例.txt
  - 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）.txt
  - 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）.txt


文件: 婚姻登记条例.txt
  解析进度: 6/6 章节, 已收集 27 条
  入库: 27/27
  完成: 清除 0 条旧记录, 插入 27 条

文件: 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）.txt
  解析进度: 1/1 章节, 已收集 90 条
  入库: 90/90
  完成: 清除 0 条旧记录, 插入 90 条

文件: 最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）.txt
  解析进度: 1/1 章节, 已收集 22 条
  入库: 22/22
  完成: 清除 0 条旧记录, 插入 22 条


In [2]:
import torch

print(torch.__version__)  # 不报错就说明 OK


OSError: [WinError 127] 找不到指定的程序。 Error loading "f:\Anaconda_env\lawApp_langGraph\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.